# 05.02 - INTRODUCING SCIKIT-LEARN - EXERCISES (SOLUTIONS)

These  short exercises mirror the basic pipeline shown in the slides:
- build **X** (feature matrix) and **y** (target array)
- choose a simple model (`LinearRegression`)
- **fit** and **predict**
- try **normalization** (`StandardScaler`, `MinMaxScaler`)

> Dataset: we'll use the classic **iris** dataset, treating `petal_length` as a *regression* target and the other numeric columns as predictors.


In [1]:

# Setup: imports
import pandas as pd
import numpy as np

# We'll use seaborn *only* to load the iris dataset because it includes convenient column names.
# If seaborn isn't available, this will raise an ImportError.
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error



## Exercise 1 — Build `X` and `y` from a labeled dataset

**Target:** `petal_length`

**Predictors:** `sepal_width`, `sepal_length`, `petal_length`


**Tasks:**
1. Load the
2.
3.
4.
5. dataset as a DataFrame.

2. Create `X` as a **2‑D table** with the three predictors and `y` as a **1‑D vector** with the target.

3. Print the shapes of `X` and `y` and show the first 5 rows of `X`.



In [2]:

iris = sns.load_dataset("iris")
X = iris[['sepal_width', 'sepal_length', 'petal_length']]  # 2-D feature matrix
y = iris['petal_length']                                   # 1-D target vector

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()


X shape: (150, 3)
y shape: (150,)


,sepal_width,sepal_length,petal_length
0,3.5,5.1,1.4
1,3.0,4.9,1.4
2,3.2,4.7,1.3
3,3.1,4.6,1.5
4,3.6,5.0,1.4



## Exercise 2 — `fit`/`predict` with `LinearRegression`

**Tasks:**
1. Create `LinearRegression(fit_intercept=True)` and print the model and hyperparameters.

2. Print the selected hyperparameters via `get_params()`.

3. **Fit** the model on `X, y` and then print `intercept_` and `coef_`. What do they mean?

4. Make predictions on the tiny `X_pred` DataFrame.



In [3]:
model = LinearRegression(fit_intercept=True)
print("Model:", model)
print("Hyperparameters:", model.get_params())

model.fit(X, y)
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

### Coefficients: [ 0.75 -1.20  0.33]

# This means:
#- Feature 1 has a positive effect (0.75)
#- Feature 2 has a negative effect (-1.20)
#- Feature 3 has a mild positive effect (0.33)

X_pred = pd.DataFrame({
    'sepal_width': [3.0, 2.5, 3.2],
    'sepal_length': [5.1, 6.0, 5.5],
    'petal_length': [0.2, 1.5, 0.4]
})
y_pred = model.predict(X_pred)
pd.DataFrame({'predicted_petal_length': y_pred})

Model: LinearRegression()
Hyperparameters: {'copy_X': True, 'fit_intercept': True, 'n_jobs': None, 'positive': False, 'tol': 1e-06}
Intercept: 3.552713678800501e-15
Coefficients: [-7.92204364e-16 -1.11022302e-16  1.00000000e+00]


,predicted_petal_length
0,0.2
1,1.5
2,0.4



## Exercise 3 — Normalisation (`StandardScaler` vs `MinMaxScaler`)

**Tasks:**
1. Fit `LinearRegression` **without** scaling; store its R² on the full data.

2. Scale `X` with **StandardScaler**, fit a new `LinearRegression`, and compute R².

3. Repeat using **MinMaxScaler**.

4. Use `inverse_transform` once to bring a scaled row back to original units.

In [20]:

# Unscaled
lr_plain = LinearRegression()
lr_plain.fit(X, y)
r2_plain = r2_score(y, lr_plain.predict(X))

# StandardScaler
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X)
lr_std = LinearRegression().fit(X_std, y)
r2_std = r2_score(y, lr_std.predict(X_std))

# MinMaxScaler
scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(X)
lr_mm = LinearRegression().fit(X_mm, y)
r2_mm = r2_score(y, lr_mm.predict(X_mm))

print(f"R² (plain): {r2_plain:.4f}")
print(f"R² (StandardScaler): {r2_std:.4f}")
print(f"R² (MinMaxScaler): {r2_mm:.4f}")

# Inverse transform demo on the first row
first_row_scaled = X_std[0:1, :]           # 2-D slice
first_row_back = scaler_std.inverse_transform(first_row_scaled)
print("First row (original):", X.iloc[0].to_list())
print("First row (back from scaled):", first_row_back.ravel().tolist())

# ravel()
# Flattens the array into 1D.
# Unlike .flatten(), it returns a view (not a copy), so it's more memory-efficient



R² (plain): 1.0000
R² (StandardScaler): 1.0000
R² (MinMaxScaler): 1.0000
First row (original): [3.5, 5.1, 1.4]
First row (back from scaled): [3.5, 5.1, 1.4]



## Exercise 4 — Holdout evaluation (`train_test_split`)

**Goal:** Estimate generalization with a simple train/test split.


**Tasks:**
1. Split `X, y` into train/test (e.g., 80/20) with a fixed `random_state`.

2. Fit `LinearRegression` on the train set.

3. Report **R²** and **MAE** on the **test** set.

4. Compare the holdout R² to your Exercise 3 numbers.



In [21]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = LinearRegression().fit(X_train, y_train)

y_hat = lr.predict(X_test)
print("Test R²:", r2_score(y_test, y_hat))
print("Test MAE:", mean_absolute_error(y_test, y_hat))


Test R²: 1.0
Test MAE: 4.144832625267251e-16



## Exercise 5 — Cross‑validation with a `Pipeline`

**Goal:** Use the same **fit/predict** interface but bundle preprocessing + model, then estimate performance via **cross‑validation**.


**Tasks:**
1. Create a `Pipeline` with `('scale', StandardScaler())` and `('lr', LinearRegression())`.

2. Use `cross_val_score` with `cv=5` (5‑fold) and `scoring='r2'`.

3. Report the individual fold scores and their mean.



In [22]:

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('lr', LinearRegression())
])

scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
print("Fold R² scores:", np.round(scores, 4))
print("Mean R²:", scores.mean().round(4))


Fold R² scores: [1. 1. 1. 1. 1.]
Mean R²: 1.0
